# BIO-NN: Run All Experiments

This notebook runs all experiments in sequence:
1. Setup & Installation
2. Train SNN on MNIST
3. Compare all neuron models
4. Criticality & emergence analysis
5. Full visualization dashboard

**Runtime:** ~15-20 minutes on T4 GPU

---
## PHASE 1: Setup

In [ ]:
!git clone https://github.com/aryaanchavan1-commits/Biological_Neural_Network_Ai.git
%cd Biological_Neural_Network_Ai
!pip install -q torch torchvision numpy scipy scikit-learn networkx matplotlib seaborn pyyaml psutil pandas
!pip install -q -e .

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

---
## PHASE 2: Train SNN on MNIST

In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from bio_nn.core.model_builder import build_model
from bio_nn.training.engine import TrainingEngine

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_data = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST('./data', train=False, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64)
print(f"MNIST loaded: {len(train_data)} train, {len(test_data)} test")

In [ ]:
config = {
    "model": {
        "type": "bio_nn",
        "time_steps": 15,
        "input_size": 784,
        "n_classes": 10,
        "layers": [{"type": "lif_neuron", "size": 256}, {"type": "lif_neuron", "size": 128}]
    },
    "neuron": {"model": "lif", "tau_mem": 20.0, "threshold": 1.0},
    "synapse": {"type": "static"},
    "encoder": {"type": "rate"},
    "decoder": {"type": "rate"},
}
model = build_model(config).to(device)
print(f"Model: {sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
engine = TrainingEngine(model, {"epochs": 15, "lr": 1e-3, "timestep": 15, "log_interval": 5}, device)
results = engine.train(train_loader, test_loader=test_loader)
print(f"\nTest Accuracy: {results['test_results']['accuracy']:.4f}")

In [ ]:
import matplotlib.pyplot as plt
epochs = [h['epoch'] for h in results['history']]
val_accs = [h['val_accuracy'] for h in results['history']]
losses = [h['train_loss'] for h in results['history']]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, val_accs, 'b-o', markersize=4)
ax1.set_title('Validation Accuracy')
ax1.set_xlabel('Epoch')
ax1.grid(True, alpha=0.3)
ax2.plot(epochs, losses, 'r-o', markersize=4)
ax2.set_title('Training Loss')
ax2.set_xlabel('Epoch')
ax2.grid(True, alpha=0.3)
plt.suptitle('MNIST Training', fontweight='bold')
plt.tight_layout()
plt.savefig('mnist_results.png', dpi=150)
plt.show()

---
## PHASE 3: Neuron Comparison

In [ ]:
import time
import pandas as pd
from bio_nn.neurons import create_neuron

neuron_models = ["lif", "adaptive_lif", "izhikevich", "dual_lif", "adex", "resonate_fire", "spiking_brain"]
bench_results = []

for name in neuron_models:
    try:
        neuron = create_neuron(name, 256).to(device)
        state = neuron._get_initial_state(32, device)
        x = torch.randn(32, 256).to(device)
        for _ in range(5): spikes, mem, state = neuron(x, state)
        start = time.time()
        for _ in range(100): spikes, mem, state = neuron(x, state)
        elapsed = time.time() - start
        bench_results.append({
            "Model": name,
            "Spike Rate": f"{spikes.float().mean().item():.4f}",
            "Time": f"{elapsed:.3f}s",
            "Steps/s": f"{100/elapsed:.0f}"
        })
    except Exception as e:
        bench_results.append({"Model": name, "Error": str(e)[:40]})

print(pd.DataFrame(bench_results).to_string(index=False))

---
## PHASE 4: Criticality Analysis

In [ ]:
from bio_nn.emergence.criticality import CriticalityDetector
from bio_nn.emergence.complexity import ComplexityMeasures

neuron = create_neuron("adaptive_lif", 512).to(device)
state = neuron._get_initial_state(16, device)
spike_log, mem_log = [], []

for i in range(200):
    x = torch.randn(16, 512).to(device) * (0.3 + 0.7 * (i / 200))
    spikes, mem, state = neuron(x, state)
    spike_log.append(spikes.cpu())
    mem_log.append(mem.cpu())

spike_tensor = torch.stack(spike_log)
print(f"Spike data: {spike_tensor.shape}")

In [ ]:
detector = CriticalityDetector(window_size=50)
branching = detector.compute_branching_ratio(spike_tensor)
complexity = ComplexityMeasures()
lyapunov = complexity.lyapunov_exponent(spike_tensor, window=50)

print(f"Branching Ratio: {branching:.4f} (target ~1.0)")
print(f"Lyapunov Exponent: {lyapunov:.6f} (target ~0)")

In [ ]:
import numpy as np
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(spike_tensor[:, 0, :128].numpy().T, aspect='auto', cmap='hot')
axes[0].set_title("Spike Raster")

rates = [s.float().mean().item() for s in spike_log]
axes[1].plot(rates, 'g-')
axes[1].axhline(y=np.mean(rates), color='r', linestyle='--')
axes[1].set_title("Firing Rate")

mem = torch.stack(mem_log)
axes[2].plot(mem[:, 0, 0].numpy(), 'b-')
axes[2].set_title("Membrane Potential")

plt.suptitle(f"Criticality: sigma={branching:.4f}, lyapunov={lyapunov:.6f}", fontweight='bold')
plt.tight_layout()
plt.savefig('criticality_results.png', dpi=150)
plt.show()

---
## PHASE 5: Full Visualization

In [ ]:
viz_types = ["lif", "adaptive_lif", "izhikevich", "dual_lif", "adex"]
viz_data = {}

for name in viz_types:
    neuron = create_neuron(name, 128).to(device)
    state = neuron._get_initial_state(8, device)
    spikes_l, mem_l = [], []
    for t in range(100):
        x = torch.randn(8, 128).to(device) * 0.8
        s, m, state = neuron(x, state)
        spikes_l.append(s.cpu())
        mem_l.append(m.cpu())
    viz_data[name] = {"spikes": torch.stack(spikes_l), "membrane": torch.stack(mem_l)}

n = len(viz_data)
fig, axes = plt.subplots(n, 2, figsize=(14, 3 * n))
for i, (name, data) in enumerate(viz_data.items()):
    axes[i, 0].imshow(data["spikes"][:, 0, :64].numpy().T, aspect='auto', cmap='hot')
    axes[i, 0].set_ylabel(name, fontweight='bold')
    if i == 0: axes[i, 0].set_title("Spike Raster")
    for j in range(3):
        axes[i, 1].plot(data["membrane"][:, 0, j].numpy(), linewidth=0.8)
    axes[i, 1].set_xlim([0, 100])
    if i == 0: axes[i, 1].set_title("Membrane")

plt.suptitle("Neuron Dynamics", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('neuron_dynamics.png', dpi=150)
plt.show()

---
## Download All Results

In [ ]:
from google.colab import files
for f in ['mnist_results.png', 'criticality_results.png', 'neuron_dynamics.png']:
    try:
        files.download(f)
    except:
        print(f"{f} not found")